# RAG experiments — interactive scratchpad

For quick, throwaway iteration on retrieval/generation parameters (top_k, chunking strategy, corpus, contextual headers on/off, query decomposition, chain-of-thought) without writing a new `eval/*.py` script each time.

**Rules this notebook still has to respect** (see `DECISIONS.md`):
1. Groq is for *generation* only — never used here to judge or score an answer.
2. Any LLM-judgment task (scoring answer quality, deciding relevance) still has to go through the external-Sonnet-5 export pattern (`eval/export_*_for_external_judge.py` + a self-contained instructions `.md`) — don't improvise a live judge call in here.
3. Nothing in this notebook commits or pushes to git.

**Known blocker as of writing:** Qdrant Cloud connectivity has been failing (`WinError 10054` / TLS reset) from this machine — every cell that calls `retriever.retrieve(...)` will raise until that's resolved. The rest of the notebook (config wiring, helper functions) is still worth having in place for when it's back.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from app.core import config
from eval.experiment_lib import (
    list_corpora,
    load_decomposed_questions,
    load_golden_set,
    run_variant,
    set_corpus,
    summarize_runs,
)

pd.set_option("display.max_colwidth", 120)
list_corpora()

## 1. Pick a corpus

Re-run this cell any time you want to switch corpus/chunking strategy — it rebuilds the retriever against the new Qdrant collection. `set_corpus` mirrors what `.env` / `DEPLOYMENT.md` document for each named config.

In [ ]:
# Edit this line to switch corpus. See list_corpora() output above for the valid names.
retriever = set_corpus("ai_act_corpus_recursive_ctxheaders")

print("active collection:", config.collection_name())
print("chunking strategy:", config.CHUNKING_STRATEGY)
print("contextual headers:", config.USE_CONTEXTUAL_HEADERS, config.CONTEXTUAL_HEADERS_PATH)

## 2. Single-question probe

The fastest loop for "does this one thing help on this one question": edit `QUESTION` / `SUB_QUESTIONS` / the toggles, re-run. `sub_questions=None` retrieves once for `QUESTION` as-is; pass a list to decompose.

In [ ]:
QUESTION = "What transparency obligations apply to a chatbot, and how do they interact with GDPR's own transparency requirements?"
SUB_QUESTIONS = None  # e.g. ["What transparency obligations does the AI Act impose on chatbots?", "What transparency requirements does GDPR impose on automated interactions?"]
USE_COT = False
TOP_K = 5

result = run_variant(
    retriever,
    question=QUESTION,
    sub_questions=SUB_QUESTIONS,
    use_cot=USE_COT,
    top_k=TOP_K,
)

print(f"chunks retrieved: {result.n_chunks}   precision/recall: {result.precision_recall}")
print(f"sources cited: {result.answer.citations}")
if USE_COT:
    print(f"\n--- reasoning ---\n{result.answer.reasoning}")
print(f"\n--- answer ---\n{result.answer.text}")

## 3. Batch comparison across variants

Runs `baseline` / `cot` / `decomposed` / `decomposed_cot` for every question in `eval/decomposed_questions.jsonl` (the 3 multi-hop questions that already have externally-generated sub-questions — see `DECOMPOSITION_INSTRUCTIONS.md`) and lays the metrics out as a table. This is the notebook version of `eval/decomposition_cot_experiment.py` — use the script for a reproducible run to paste into `EVALUATION_HISTORY.md`, use this cell for poking at intermediate results interactively (e.g. `runs[i].answer.text` for any row).

In [ ]:
golden_set = load_golden_set()
decomposed = load_decomposed_questions()

runs = []
for question, sub_questions in decomposed.items():
    for label, use_cot, sqs in [
        ("baseline", False, None),
        ("cot", True, None),
        ("decomposed", False, sub_questions),
        ("decomposed_cot", True, sub_questions),
    ]:
        runs.append(
            run_variant(
                retriever,
                question=question,
                sub_questions=sqs,
                use_cot=use_cot,
                variant_label=label,
                golden_set=golden_set,
            )
        )

summarize_runs(runs)

## 4. Quick parameter sweep

Example: how does `top_k` affect precision/recall for one question, without decomposition or CoT. Swap in whatever you're actually tuning (e.g. `config.MIN_OVERLAP_FRACTION` isn't live-editable this way since `span_scoring.py` reads it as a module constant — edit that constant directly and re-import if you need to sweep it too).

In [ ]:
sweep_question = next(iter(decomposed))  # first decomposed question, or set your own
sweep_runs = [
    run_variant(retriever, question=sweep_question, top_k=k, variant_label=f"top_k={k}", golden_set=golden_set)
    for k in [3, 5, 8, 12]
]
summarize_runs(sweep_runs)